# DAY 20 — Robustness + Ablation Study
**Underwater Garbage Detection**

This notebook performs Day 20 only:
- augmentation/component ablation using already-trained checkpoints
- robustness analysis for low visibility and small objects
- optional clutter/occlusion proxy analysis from the existing validation labels
- the same detection metrics across conditions
- inference-time/FPS measurement
- saved robustness/ablation tables and research insight

**Important:** No training is performed here and the final test set is never used. All selection/analysis uses the existing validation split only.

In [2]:
# ============================================================
# DAY 20 — FIND THE ACTUAL DATASET ZIP
# ============================================================

from pathlib import Path
from google.colab import drive
import zipfile

drive.mount("/content/drive", force_remount=False)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/INTERNSHIPWORK"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

print("=" * 70)
print("SEARCHING FOR DATASET FILES")
print("=" * 70)

found_zips = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    print("\nSearching:", root)

    for p in root.rglob("*.zip"):
        print("ZIP:", p)
        found_zips.append(p)

print("\n" + "=" * 70)

if not found_zips:
    print("❌ No ZIP files found.")
else:
    print("ZIP files found:", len(found_zips))

    for i, p in enumerate(found_zips):
        print(f"\n[{i}] {p}")

        try:
            with zipfile.ZipFile(p, "r") as z:
                names = z.namelist()

                print("Files inside:", len(names))

                # Show likely dataset structure
                interesting = [
                    x for x in names
                    if (
                        "instance" in x.lower()
                        or "material" in x.lower()
                        or "train" in x.lower()
                        or "val" in x.lower()
                        or "annotation" in x.lower()
                    )
                ]

                for x in interesting[:30]:
                    print("   ", x)

        except Exception as e:
            print("Could not inspect:", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SEARCHING FOR DATASET FILES

Searching: /content/drive/MyDrive/INTERNSHIPWORK
ZIP: /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip

Searching: /content/drive/MyDrive
ZIP: /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip

Searching: /content
ZIP: /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip

ZIP files found: 3

[0] /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip
Files inside: 28868
    dataset/instance_version/
    dataset/instance_version/README.txt
    dataset/instance_version/instances_val_trashcan.json
    dataset/instance_version/instances_train_trashcan.json
    dataset/material_version/
    dataset/material_version/README.txt
    dataset/material_version/instances_val_trashcan.json
    dataset/material_version/instances_train_trashcan.json
    dataset/instance_version/val/
    dataset/instance_version/val/vid_000142_frame0000037.jpg
    datase

In [4]:
# ============================================================
# DAY 20 — COMPLETE DATASET LOCATOR
# ============================================================

from pathlib import Path
from google.colab import drive
import zipfile

drive.mount("/content/drive", force_remount=False)

print("=" * 75)
print("DAY 20 — SEARCHING FOR ORIGINAL DATA")
print("=" * 75)

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/INTERNSHIPWORK"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

# ------------------------------------------------------------
# SEARCH FOR DATASET V1
# ------------------------------------------------------------

v1_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    try:
        for p in root.rglob("data.yaml"):

            parent = p.parent

            if (
                (parent / "images" / "val").exists()
                and
                (parent / "labels" / "val").exists()
            ):
                v1_candidates.append(parent)

    except Exception as e:
        print("Search warning:", e)

print("\nDataset V1 candidates:")

if v1_candidates:
    for p in v1_candidates:
        print("✅", p)
else:
    print("❌ No complete Dataset V1 found.")

# ------------------------------------------------------------
# SEARCH FOR ZIP FILES
# ------------------------------------------------------------

zip_candidates = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    try:
        for p in root.rglob("*.zip"):
            zip_candidates.append(p)
    except Exception:
        pass

print("\n" + "=" * 75)
print("ZIP FILES")
print("=" * 75)

if not zip_candidates:
    print("❌ No ZIP files found.")

else:

    for i, z in enumerate(zip_candidates):

        print(f"\n[{i}] {z}")

        try:

            with zipfile.ZipFile(z, "r") as archive:

                names = archive.namelist()

                print("    Files:", len(names))

                # Show first 20 paths
                for name in names[:20]:
                    print("    ", name)

                # Detect important structures
                has_images = any(
                    "images/" in n.lower()
                    for n in names
                )

                has_annotations = any(
                    "annotation" in n.lower()
                    for n in names
                )

                has_instance = any(
                    "instance" in n.lower()
                    for n in names
                )

                print("    images:", has_images)
                print("    annotations:", has_annotations)
                print("    instance:", has_instance)

        except Exception as e:
            print("    Could not inspect ZIP:", e)

print("\n" + "=" * 75)
print("SEARCH FINISHED")
print("=" * 75)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DAY 20 — SEARCHING FOR ORIGINAL DATA

Dataset V1 candidates:
❌ No complete Dataset V1 found.

ZIP FILES

[0] /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip
    Files: 28868
     dataset/
     dataset/README.txt
     dataset/instance_version/
     dataset/instance_version/README.txt
     dataset/instance_version/instances_val_trashcan.json
     dataset/instance_version/instances_train_trashcan.json
     dataset/material_version/
     dataset/material_version/README.txt
     dataset/material_version/instances_val_trashcan.json
     dataset/material_version/instances_train_trashcan.json
     dataset/original_data/
     dataset/original_data/README.txt
     dataset/scripts/
     dataset/scripts/trash_can_coco.py
     dataset/instance_version/val/
     dataset/instance_version/val/vid_000142_frame0000037.jpg
     dataset/instance_version/val/vid_000028_frame000

In [6]:
# ============================================================
# DAY 20 — CELL 1
# RESTORE RAW DATA + BUILD YOLO DATASET
# ============================================================

from pathlib import Path
import os
import json
import zipfile
import shutil
import yaml
import random

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------
# CORRECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs"
)

SOURCE_ZIP = Path(
    "/content/drive/MyDrive/INTERNSHIPWORK/dataset.zip"
)

RAW_ROOT = Path(
    "/content/TrashCAN_raw"
)

DATASET_ROOT = Path(
    "/content/TrashCAN_YOLO_Dataset_V1"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

DAY20_ROOT = PROJECT_DIR / "day20_robustness"
DAY20_ROOT.mkdir(parents=True, exist_ok=True)

DAY17_BEST = (
    PROJECT_DIR /
    "day17_yolov8n_tuned_continuation" /
    "weights" /
    "best.pt"
)

print("=" * 70)
print("DAY 20 — PATH SETUP")
print("=" * 70)

print("PROJECT_DIR :", PROJECT_DIR)
print("SOURCE_ZIP  :", SOURCE_ZIP)
print("RAW_ROOT    :", RAW_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("DATA_YAML   :", DATA_YAML)
print("DAY20_ROOT  :", DAY20_ROOT)
print("DAY17_BEST  :", DAY17_BEST)

print("\nExistence:")
print("PROJECT_DIR :", PROJECT_DIR.exists())
print("SOURCE_ZIP  :", SOURCE_ZIP.exists())
print("DAY17_BEST  :", DAY17_BEST.exists())

if not SOURCE_ZIP.exists():
    raise FileNotFoundError(
        f"Your actual dataset ZIP was not found:\n{SOURCE_ZIP}"
    )

if not DAY17_BEST.exists():
    raise FileNotFoundError(
        f"Day 17 best.pt was not found:\n{DAY17_BEST}"
    )


# ============================================================
# EXTRACT RAW DATA
# ============================================================

if not RAW_ROOT.exists():

    print("\nExtracting dataset.zip ...")

    RAW_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
        z.extractall(RAW_ROOT)

    print("✅ Raw dataset extracted.")

else:
    print("\n✅ Raw dataset already extracted.")


# Find instance_version automatically
INSTANCE_ROOT = None

candidates = [
    RAW_ROOT / "dataset" / "instance_version",
    RAW_ROOT / "instance_version",
]

for p in candidates:
    if (
        p.exists()
        and (p / "train").exists()
        and (p / "val").exists()
    ):
        INSTANCE_ROOT = p
        break


if INSTANCE_ROOT is None:

    found = list(
        RAW_ROOT.glob("**/instance_version")
    )

    for p in found:
        if (
            (p / "train").exists()
            and (p / "val").exists()
        ):
            INSTANCE_ROOT = p
            break


if INSTANCE_ROOT is None:
    raise FileNotFoundError(
        "Could not find instance_version/train and instance_version/val "
        "inside the extracted dataset."
    )


TRAIN_RAW = INSTANCE_ROOT / "train"
VAL_RAW = INSTANCE_ROOT / "val"

TRAIN_JSON = INSTANCE_ROOT / "instances_train_trashcan.json"
VAL_JSON = INSTANCE_ROOT / "instances_val_trashcan.json"

print("\nRAW DATA FOUND")
print("INSTANCE_ROOT:", INSTANCE_ROOT)
print("TRAIN_RAW    :", TRAIN_RAW)
print("VAL_RAW      :", VAL_RAW)
print("TRAIN_JSON   :", TRAIN_JSON)
print("VAL_JSON     :", VAL_JSON)

if not TRAIN_JSON.exists():
    raise FileNotFoundError(TRAIN_JSON)

if not VAL_JSON.exists():
    raise FileNotFoundError(VAL_JSON)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DAY 20 — PATH SETUP
PROJECT_DIR : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs
SOURCE_ZIP  : /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip
RAW_ROOT    : /content/TrashCAN_raw
DATASET_ROOT: /content/TrashCAN_YOLO_Dataset_V1
DATA_YAML   : /content/TrashCAN_YOLO_Dataset_V1/data.yaml
DAY20_ROOT  : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness
DAY17_BEST  : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/best.pt

Existence:
PROJECT_DIR : True
SOURCE_ZIP  : True
DAY17_BEST  : True

Extracting dataset.zip ...
✅ Raw dataset extracted.

RAW DATA FOUND
INSTANCE_ROOT: /content/TrashCAN_raw/dataset/instance_version
TRAIN_RAW    : /content/TrashCAN_raw/dataset/instance_version/train
VAL_RAW      : /content/TrashCAN_raw/dataset/instance_version/val
TRAIN_JSON   : /content/TrashCAN_raw/dataset/instance_versi

In [7]:
# ============================================================
# DAY 20 — FIXED SETUP + DATASET RESTORE + MODEL LOAD
# RUN THIS CELL AS ONE COMPLETE CELL
# ============================================================

from pathlib import Path
import os
import sys
import json
import zipfile
import shutil
import yaml
import subprocess

from google.colab import drive

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)


# ------------------------------------------------------------
# 2. CORRECT PATHS FROM YOUR PROJECT
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs"
)

SOURCE_ZIP = Path(
    "/content/drive/MyDrive/INTERNSHIPWORK/dataset.zip"
)

RAW_ROOT = Path(
    "/content/TrashCAN_raw"
)

DATASET_ROOT = Path(
    "/content/TrashCAN_YOLO_Dataset_V1"
)

DATA_YAML = DATASET_ROOT / "data.yaml"

DAY20_ROOT = PROJECT_DIR / "day20_robustness"

DAY17_BEST = (
    PROJECT_DIR /
    "day17_yolov8n_tuned_continuation" /
    "weights" /
    "best.pt"
)

DAY20_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. PRINT PATHS
# ------------------------------------------------------------

print("=" * 70)
print("DAY 20 — PATH CHECK")
print("=" * 70)

print("PROJECT_DIR :", PROJECT_DIR)
print("SOURCE_ZIP  :", SOURCE_ZIP)
print("RAW_ROOT    :", RAW_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("DATA_YAML   :", DATA_YAML)
print("DAY20_ROOT  :", DAY20_ROOT)
print("DAY17_BEST  :", DAY17_BEST)

print("\nEXISTS:")
print("PROJECT_DIR :", PROJECT_DIR.exists())
print("SOURCE_ZIP  :", SOURCE_ZIP.exists())
print("DAY17_BEST  :", DAY17_BEST.exists())


if not SOURCE_ZIP.exists():
    raise FileNotFoundError(
        f"\nDataset ZIP not found:\n{SOURCE_ZIP}"
    )

if not DAY17_BEST.exists():
    raise FileNotFoundError(
        f"\nDay 17 best.pt not found:\n{DAY17_BEST}"
    )


# ------------------------------------------------------------
# 4. FIND / RESTORE RAW DATA
# ------------------------------------------------------------

if not RAW_ROOT.exists():

    print("\nExtracting dataset.zip...")

    RAW_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        SOURCE_ZIP,
        "r"
    ) as z:

        z.extractall(RAW_ROOT)

    print("✅ ZIP extracted.")

else:

    print("\n✅ Raw dataset already extracted.")


# ------------------------------------------------------------
# 5. FIND INSTANCE VERSION
# ------------------------------------------------------------

INSTANCE_ROOT = None

possible_instance_roots = [

    RAW_ROOT /
    "dataset" /
    "instance_version",

    RAW_ROOT /
    "instance_version",

]


for p in possible_instance_roots:

    if (
        p.exists()
        and (p / "train").exists()
        and (p / "val").exists()
    ):

        INSTANCE_ROOT = p
        break


# If not found, search recursively
if INSTANCE_ROOT is None:

    for p in RAW_ROOT.glob(
        "**/instance_version"
    ):

        if (
            (p / "train").exists()
            and (p / "val").exists()
        ):

            INSTANCE_ROOT = p
            break


if INSTANCE_ROOT is None:

    raise FileNotFoundError(
        "\nCould not find instance_version/train "
        "and instance_version/val."
    )


TRAIN_RAW = INSTANCE_ROOT / "train"
VAL_RAW = INSTANCE_ROOT / "val"

TRAIN_JSON = (
    INSTANCE_ROOT /
    "instances_train_trashcan.json"
)

VAL_JSON = (
    INSTANCE_ROOT /
    "instances_val_trashcan.json"
)


print("\nRAW DATA:")
print("INSTANCE_ROOT:", INSTANCE_ROOT)
print("TRAIN_RAW    :", TRAIN_RAW)
print("VAL_RAW      :", VAL_RAW)
print("TRAIN_JSON   :", TRAIN_JSON)
print("VAL_JSON     :", VAL_JSON)


# ------------------------------------------------------------
# 6. CREATE YOLO DATASET IF MISSING
# ------------------------------------------------------------

if not DATA_YAML.exists():

    print("\nDataset V1 not found.")
    print("Creating YOLO Dataset V1...")

    train_images_dir = (
        DATASET_ROOT /
        "images" /
        "train"
    )

    val_images_dir = (
        DATASET_ROOT /
        "images" /
        "val"
    )

    train_labels_dir = (
        DATASET_ROOT /
        "labels" /
        "train"
    )

    val_labels_dir = (
        DATASET_ROOT /
        "labels" /
        "val"
    )

    for p in [
        train_images_dir,
        val_images_dir,
        train_labels_dir,
        val_labels_dir
    ]:
        p.mkdir(
            parents=True,
            exist_ok=True
        )


    # --------------------------------------------------------
    # YOUR 8 GARBAGE CLASSES
    # --------------------------------------------------------

    CLASS_NAMES = [
        "trash_etc",
        "trash_fabric",
        "trash_fishing_gear",
        "trash_metal",
        "trash_paper",
        "trash_plastic",
        "trash_rubber",
        "trash_wood"
    ]

    CLASS_TO_ID = {
        name: i
        for i, name in enumerate(CLASS_NAMES)
    }


    # --------------------------------------------------------
    # LOAD COCO JSON
    # --------------------------------------------------------

    with open(
        TRAIN_JSON,
        "r",
        encoding="utf-8"
    ) as f:

        TRAIN_COCO = json.load(f)


    with open(
        VAL_JSON,
        "r",
        encoding="utf-8"
    ) as f:

        VAL_COCO = json.load(f)


    # --------------------------------------------------------
    # CATEGORY MAP
    # --------------------------------------------------------

    CATEGORY_MAP = {
        c["id"]: c["name"]
        for c in TRAIN_COCO["categories"]
    }


    print("\nOriginal categories:")

    for cid, name in CATEGORY_MAP.items():
        print(cid, "->", name)


    # --------------------------------------------------------
    # COCO → YOLO CONVERTER
    # --------------------------------------------------------

    from collections import defaultdict


    def convert_split(
        coco,
        source_dir,
        image_out,
        label_out,
        split_name
    ):

        images_by_id = {
            img["id"]: img
            for img in coco["images"]
        }

        anns_by_image = defaultdict(list)

        for ann in coco["annotations"]:

            if ann.get(
                "iscrowd",
                0
            ) == 1:
                continue

            anns_by_image[
                ann["image_id"]
            ].append(ann)


        copied = 0
        boxes = 0
        missing = 0


        for image_id, info in images_by_id.items():

            file_name = Path(
                info["file_name"]
            ).name

            source_file = (
                source_dir /
                file_name
            )

            if not source_file.exists():

                matches = list(
                    source_dir.rglob(
                        file_name
                    )
                )

                if not matches:

                    missing += 1
                    continue

                source_file = matches[0]


            destination = (
                image_out /
                file_name
            )

            shutil.copy2(
                source_file,
                destination
            )

            copied += 1


            width = info["width"]
            height = info["height"]

            label_lines = []


            for ann in anns_by_image.get(
                image_id,
                []
            ):

                category_name = CATEGORY_MAP.get(
                    ann["category_id"]
                )

                if category_name not in CLASS_TO_ID:
                    continue

                x, y, w, h = ann["bbox"]

                if w <= 0 or h <= 0:
                    continue


                # COCO → YOLO
                xc = (
                    x + w / 2
                ) / width

                yc = (
                    y + h / 2
                ) / height

                wn = w / width
                hn = h / height


                # Keep values valid
                xc = max(
                    0,
                    min(1, xc)
                )

                yc = max(
                    0,
                    min(1, yc)
                )

                wn = max(
                    0,
                    min(1, wn)
                )

                hn = max(
                    0,
                    min(1, hn)
                )


                class_id = CLASS_TO_ID[
                    category_name
                ]


                label_lines.append(
                    f"{class_id} "
                    f"{xc:.6f} "
                    f"{yc:.6f} "
                    f"{wn:.6f} "
                    f"{hn:.6f}"
                )

                boxes += 1


            label_path = (
                label_out /
                f"{Path(file_name).stem}.txt"
            )

            with open(
                label_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(
                    "\n".join(label_lines)
                )


        print(
            f"\n{split_name.upper()}:"
        )

        print(
            "Images copied:",
            copied
        )

        print(
            "Boxes:",
            boxes
        )

        print(
            "Missing:",
            missing
        )


    # --------------------------------------------------------
    # CONVERT TRAIN + VAL
    # --------------------------------------------------------

    convert_split(
        TRAIN_COCO,
        TRAIN_RAW,
        train_images_dir,
        train_labels_dir,
        "train"
    )

    convert_split(
        VAL_COCO,
        VAL_RAW,
        val_images_dir,
        val_labels_dir,
        "val"
    )


    # --------------------------------------------------------
    # CREATE data.yaml
    # --------------------------------------------------------

    DATA_CONFIG = {

        "path": str(DATASET_ROOT),

        "train": "images/train",

        "val": "images/val",

        "nc": 8,

        "names": {
            0: "trash_etc",
            1: "trash_fabric",
            2: "trash_fishing_gear",
            3: "trash_metal",
            4: "trash_paper",
            5: "trash_plastic",
            6: "trash_rubber",
            7: "trash_wood"
        }
    }


    with open(
        DATA_YAML,
        "w",
        encoding="utf-8"
    ) as f:

        yaml.safe_dump(
            DATA_CONFIG,
            f,
            sort_keys=False
        )


    print(
        "\n✅ Dataset V1 created."
    )

else:

    print(
        "\n✅ Dataset V1 already exists."
    )


# ------------------------------------------------------------
# 7. VERIFY DATASET
# ------------------------------------------------------------

if not DATA_YAML.exists():

    raise FileNotFoundError(
        f"Dataset YAML still missing:\n{DATA_YAML}"
    )


VAL_IMG_DIR = (
    DATASET_ROOT /
    "images" /
    "val"
)

VAL_LABEL_DIR = (
    DATASET_ROOT /
    "labels" /
    "val"
)

TRAIN_IMG_DIR = (
    DATASET_ROOT /
    "images" /
    "train"
)


if not VAL_IMG_DIR.exists():

    raise FileNotFoundError(
        f"Validation images missing:\n{VAL_IMG_DIR}"
    )


if not VAL_LABEL_DIR.exists():

    raise FileNotFoundError(
        f"Validation labels missing:\n{VAL_LABEL_DIR}"
    )


# ------------------------------------------------------------
# 8. LOAD YAML
# ------------------------------------------------------------

with open(
    DATA_YAML,
    "r",
    encoding="utf-8"
) as f:

    DATA_CFG = yaml.safe_load(f)


CLASS_NAMES = DATA_CFG["names"]

if isinstance(
    CLASS_NAMES,
    dict
):

    CLASS_NAMES = [
        CLASS_NAMES[k]
        for k in sorted(
            CLASS_NAMES,
            key=lambda x: int(x)
        )
    ]


# ------------------------------------------------------------
# 9. VALIDATION IMAGES
# ------------------------------------------------------------

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


VAL_IMAGES = sorted([
    p
    for p in VAL_IMG_DIR.iterdir()
    if (
        p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTS
    )
])


# ------------------------------------------------------------
# 10. INSTALL / LOAD YOLO
# ------------------------------------------------------------

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "ultralytics==8.4.165"
])


from ultralytics import YOLO


MODEL = YOLO(
    str(DAY17_BEST)
)


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("✅ DAY 20 SETUP COMPLETE")
print("=" * 70)

print("DATASET_ROOT :", DATASET_ROOT)
print("DATA_YAML    :", DATA_YAML)
print("TRAIN images :", len(list(TRAIN_IMG_DIR.iterdir())))
print("VAL images   :", len(VAL_IMAGES))
print("VAL labels   :", len(list(VAL_LABEL_DIR.iterdir())))
print("DAY17 MODEL  :", DAY17_BEST)
print("CLASSES      :", CLASS_NAMES)

print("\nTEST SET WILL NOT BE USED.")
print("You can now continue with the Day 20 robustness/ablation cells.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DAY 20 — PATH CHECK
PROJECT_DIR : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs
SOURCE_ZIP  : /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip
RAW_ROOT    : /content/TrashCAN_raw
DATASET_ROOT: /content/TrashCAN_YOLO_Dataset_V1
DATA_YAML   : /content/TrashCAN_YOLO_Dataset_V1/data.yaml
DAY20_ROOT  : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness
DAY17_BEST  : /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/best.pt

EXISTS:
PROJECT_DIR : True
SOURCE_ZIP  : True
DAY17_BEST  : True

✅ Raw dataset already extracted.

RAW DATA:
INSTANCE_ROOT: /content/TrashCAN_raw/dataset/instance_version
TRAIN_RAW    : /content/TrashCAN_raw/dataset/instance_version/train
VAL_RAW      : /content/TrashCAN_raw/dataset/instance_version/val
TRAIN_JSON   : /content/TrashCAN_raw/dataset/instance_version/instances_train_trashcan

In [8]:
# ============================================================
# 3. LOCATE EXISTING TRAINED MODELS — READ ONLY
# ============================================================
def find_best(patterns):
    candidates = []
    for pattern in patterns:
        for d in PROJECT_DIR.glob(pattern):
            p = d / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)
    if not candidates:
        return None
    # Prefer the checkpoint with the newest modification time.
    return max(candidates, key=lambda p: p.stat().st_mtime)

MODELS = {}

MODELS["Day 15 - YOLOv8n baseline"] = find_best([
    "day15_yolov8n_baseline*"
])

MODELS["Day 16 - YOLOv8n img832"] = find_best([
    "day16_yolov8n_img832*"
])

MODELS["Day 17 - YOLOv8n tuned"] = find_best([
    "day17_yolov8n_tuned_continuation",
    "day17_yolov8n_tuned*"
])

MODELS["Day 18 - YOLO11n"] = find_best([
    "day18_yolo11n_640*",
    "day18_yolo11n*"
])

print("Located checkpoints:")
for name, path in MODELS.items():
    print(f"{name}: {path}")

if not any(MODELS.values()):
    raise FileNotFoundError(
        f"No trained best.pt checkpoints were found under {PROJECT_DIR}"
    )

# Remove missing entries; the notebook evaluates only models that actually exist.
MODELS = {k:v for k,v in MODELS.items() if v is not None}

Located checkpoints:
Day 15 - YOLOv8n baseline: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day15_yolov8n_baseline-22/weights/best.pt
Day 16 - YOLOv8n img832: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day16_yolov8n_img832-28/weights/best.pt
Day 17 - YOLOv8n tuned: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/best.pt
Day 18 - YOLO11n: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day18_yolo11n_640-5/weights/best.pt


In [9]:
# ============================================================
# 4. BUILD VALIDATION CONDITION SETS
# ============================================================
# These are analysis subsets of the ORIGINAL validation split.
# No image is moved or copied and no new train/val/test split is made.

records = []
for img in VAL_IMAGES:
    im = cv2.imread(str(img))
    if im is None:
        continue

    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    brightness = float(gray.mean())
    contrast = float(gray.std())
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    label = VAL_LABEL_DIR / f"{img.stem}.txt"
    objects = []
    if label.exists():
        for line in label.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                try:
                    cls = int(parts[0])
                    _,_,w,h = map(float, parts[1:5])
                    area = w*h
                    objects.append((cls, area))
                except:
                    pass

    small = sum(a <= 0.01 for _,a in objects)
    very_small = sum(a <= 0.005 for _,a in objects)

    # Conservative image-quality proxies, not ground-truth difficulty labels.
    records.append({
        "image": str(img),
        "brightness": brightness,
        "contrast": contrast,
        "sharpness": sharpness,
        "objects": len(objects),
        "small_objects": small,
        "very_small_objects": very_small
    })

VAL_META = pd.DataFrame(records)

# Thresholds are percentile-based only for creating analysis subsets.
# They do not change the dataset or labels.
brightness_cut = VAL_META["brightness"].quantile(0.25)
contrast_cut = VAL_META["contrast"].quantile(0.25)
sharpness_cut = VAL_META["sharpness"].quantile(0.25)

VAL_META["condition"] = "normal"
VAL_META.loc[VAL_META["brightness"] <= brightness_cut, "condition"] = "low_visibility_brightness"
VAL_META.loc[VAL_META["contrast"] <= contrast_cut, "condition"] = "low_visibility_contrast"

# Small-object condition = images containing at least one bbox <= 1% area.
VAL_META.loc[VAL_META["small_objects"] > 0, "condition"] = "small_object"

# Additional separate masks are kept for reporting rather than overwriting labels.
VAL_META["low_visibility"] = (
    (VAL_META["brightness"] <= brightness_cut) |
    (VAL_META["contrast"] <= contrast_cut)
)
VAL_META["small_object_condition"] = VAL_META["small_objects"] > 0

VAL_META.to_csv(DAY20_ROOT/"validation_condition_metadata.csv", index=False)

print("Validation condition thresholds:")
print("brightness <= ", round(brightness_cut, 3))
print("contrast   <= ", round(contrast_cut, 3))
print("sharpness  <= ", round(sharpness_cut, 3))
print()
print("Low visibility images:", int(VAL_META["low_visibility"].sum()))
print("Small-object images :", int(VAL_META["small_object_condition"].sum()))

Validation condition thresholds:
brightness <=  79.408
contrast   <=  24.343
sharpness  <=  2159.049

Low visibility images: 485
Small-object images : 0


In [10]:
# ============================================================
# 5. CREATE TEMPORARY VALIDATION LISTS
# ============================================================
# Ultralytics accepts a text file containing image paths.
# Each condition remains a subset of the original validation set.

CONDITION_LISTS = {}

all_list = DAY20_ROOT/"val_all.txt"
all_list.write_text("\n".join(VAL_META["image"].tolist()))
CONDITION_LISTS["All validation"] = all_list

low_list = DAY20_ROOT/"val_low_visibility.txt"
low_list.write_text("\n".join(
    VAL_META.loc[VAL_META["low_visibility"], "image"].tolist()
))
CONDITION_LISTS["Low visibility"] = low_list

small_list = DAY20_ROOT/"val_small_object.txt"
small_list.write_text("\n".join(
    VAL_META.loc[VAL_META["small_object_condition"], "image"].tolist()
))
CONDITION_LISTS["Small objects"] = small_list

print({k: sum(1 for _ in open(v)) for k,v in CONDITION_LISTS.items()})

{'All validation': 1147, 'Low visibility': 485, 'Small objects': 0}


In [11]:
# ============================================================
# 6. CREATE CONDITION-SPECIFIC YAML FILES
# ============================================================
def make_condition_yaml(list_path, name):
    out = DAY20_ROOT / f"{name}.yaml"
    cfg = {
        "path": str(DATASET_ROOT),
        "train": "images/train",
        "val": str(list_path),
        "test": "images/test",
        "names": CLASS_NAMES
    }
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

CONDITION_YAMLS = {
    name: make_condition_yaml(path, name.lower().replace(" ","_"))
    for name,path in CONDITION_LISTS.items()
}
print(CONDITION_YAMLS)

{'All validation': PosixPath('/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness/all_validation.yaml'), 'Low visibility': PosixPath('/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness/low_visibility.yaml'), 'Small objects': PosixPath('/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness/small_objects.yaml')}


In [12]:
# ============================================================
# 7. EVALUATION FUNCTION
# ============================================================
def evaluate_model(model_path, yaml_path, condition_name):
    model = YOLO(str(model_path))

    # Validation only.
    metrics = model.val(
        data=str(yaml_path),
        split="val",
        imgsz=832,
        batch=16,
        plots=False,
        verbose=False
    )

    box = metrics.box

    # Per-class values supplied by Ultralytics.
    maps = np.asarray(box.maps, dtype=float)
    p = np.asarray(getattr(box, "p", []), dtype=float)
    r = np.asarray(getattr(box, "r", []), dtype=float)

    if p.size == 0:
        p = np.full(len(CLASS_NAMES), np.nan)
    if r.size == 0:
        r = np.full(len(CLASS_NAMES), np.nan)

    f1 = 2*p*r/(p+r+1e-12)

    # Speed values are milliseconds/image when available.
    speed = getattr(metrics, "speed", {}) or {}
    infer_ms = float(speed.get("inference", np.nan))
    fps = 1000.0/infer_ms if np.isfinite(infer_ms) and infer_ms > 0 else np.nan

    rows = []
    for i,name in enumerate(CLASS_NAMES):
        rows.append({
            "method": model_path.parent.parent.name,
            "condition": condition_name,
            "class": name,
            "precision": p[i] if i < len(p) else np.nan,
            "recall": r[i] if i < len(r) else np.nan,
            "F1": f1[i] if i < len(f1) else np.nan,
            "mAP50-95": maps[i] if i < len(maps) else np.nan,
            "inference_ms": infer_ms,
            "FPS": fps
        })

    overall = pd.DataFrame([{
        "method": model_path.parent.parent.name,
        "condition": condition_name,
        "precision": float(np.nanmean(p)),
        "recall": float(np.nanmean(r)),
        "F1": float(np.nanmean(f1)),
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
        "inference_ms": infer_ms,
        "FPS": fps
    }])

    return overall, pd.DataFrame(rows)

In [13]:
# ============================================================
# 8. RUN DAY 20 ROBUSTNESS EVALUATION
# ============================================================
overall_rows = []
class_rows = []

for method, checkpoint in MODELS.items():
    print("\nEvaluating:", method)
    print("Checkpoint:", checkpoint)

    for condition, yaml_path in CONDITION_YAMLS.items():
        print("  ->", condition)
        try:
            overall, classes = evaluate_model(checkpoint, yaml_path, condition)
            overall["method_label"] = method
            classes["method_label"] = method
            overall_rows.append(overall)
            class_rows.append(classes)
        except Exception as e:
            print("  SKIPPED:", condition, "|", repr(e))

if not overall_rows:
    raise RuntimeError("No Day 20 evaluation completed.")

OVERALL_RESULTS = pd.concat(overall_rows, ignore_index=True)
CLASS_RESULTS = pd.concat(class_rows, ignore_index=True)

OVERALL_RESULTS.to_csv(DAY20_ROOT/"day20_overall_robustness.csv", index=False)
CLASS_RESULTS.to_csv(DAY20_ROOT/"day20_classwise_robustness.csv", index=False)

print("\nDAY 20 OVERALL RESULTS")
display(OVERALL_RESULTS.round(4))


Evaluating: Day 15 - YOLOv8n baseline
Checkpoint: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day15_yolov8n_baseline-22/weights/best.pt
  -> All validation
Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 841.8±268.3 MB/s, size: 18.9 KB)
val: Scanning /content/TrashCAN_YOLO_Dataset_V1/labels/val... 1147 images, 1147 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1147/1147 3.2Kit/s 0.4s
val: New cache created: /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache
WARNING ⚠️ Labels are missing or empty in /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 4.0s/it 4:49
                   all       1147        

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:892: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/tmp/ipykernel_511/4109259802.py:53: RuntimeWarning: Mean of empty slice
  "precision": float(np.nanmean(p)),
/tmp/ipykernel_511/4109259802.py:54: RuntimeWarning: Mean of empty slice
  "recall": float(np.nanmean(r)),
/tmp/ipykernel_511/4109259802.py:55: RuntimeWarning: Mean of empty slice
  "F1": float(np.nanmean(f1)),


Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 638.2±308.1 MB/s, size: 20.6 KB)
val: Scanning /content/TrashCAN_YOLO_Dataset_V1/labels/val... 485 images, 485 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 485/485 3.6Kit/s 0.1s
val: New cache created: /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache
WARNING ⚠️ Labels are missing or empty in /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 3.7s/it 1:55
                   all        485          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 4.3ms preprocess, 226.6ms infe

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:892: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/tmp/ipykernel_511/4109259802.py:53: RuntimeWarning: Mean of empty slice
  "precision": float(np.nanmean(p)),
/tmp/ipykernel_511/4109259802.py:54: RuntimeWarning: Mean of empty slice
  "recall": float(np.nanmean(r)),
/tmp/ipykernel_511/4109259802.py:55: RuntimeWarning: Mean of empty slice
  "F1": float(np.nanmean(f1)),


Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
  SKIPPED: Small objects | FileNotFoundError('\x1b\x1bval: \x1bError loading data from /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness/val_small_object.txt\nSee https://docs.ultralytics.com/datasets for dataset formatting guidance.')

Evaluating: Day 16 - YOLOv8n img832
Checkpoint: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day16_yolov8n_img832-28/weights/best.pt
  -> All validation
Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 643.5±185.7 MB/s, size: 21.7 KB)
val: Scanning /content/TrashCAN_YOLO_Dataset_V1/labels/val... 1147 images, 1147 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1147/1147 3.5Kit/s 0.3s
val: New cache created: /content/

...

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:892: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/tmp/ipykernel_511/4109259802.py:53: RuntimeWarning: Mean of empty slice
  "precision": float(np.nanmean(p)),
/tmp/ipykernel_511/4109259802.py:54: RuntimeWarning: Mean of empty slice
  "recall": float(np.nanmean(r)),
/tmp/ipykernel_511/4109259802.py:55: RuntimeWarning: Mean of empty slice
  "F1": float(np.nanmean(f1)),


Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1040.9±256.1 MB/s, size: 28.4 KB)
val: Scanning /content/TrashCAN_YOLO_Dataset_V1/labels/val... 485 images, 485 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 485/485 3.5Kit/s 0.1s
val: New cache created: /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache
WARNING ⚠️ Labels are missing or empty in /content/TrashCAN_YOLO_Dataset_V1/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 3.9s/it 1:59
                   all        485          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 7.5ms preprocess, 232.1ms 

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/metrics.py:892: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:139: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/tmp/ipykernel_511/4109259802.py:53: RuntimeWarning: Mean of empty slice
  "precision": float(np.nanmean(p)),
/tmp/ipykernel_511/4109259802.py:54: RuntimeWarning: Mean of empty slice
  "recall": float(np.nanmean(r)),
/tmp/ipykernel_511/4109259802.py:55: RuntimeWarning: Mean of empty slice
  "F1": float(np.nanmean(f1)),


YOLO11n summary (fused): 100 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
  SKIPPED: Small objects | FileNotFoundError('\x1b\x1bval: \x1bError loading data from /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day20_robustness/val_small_object.txt\nSee https://docs.ultralytics.com/datasets for dataset formatting guidance.')

DAY 20 OVERALL RESULTS


,method,condition,precision,recall,F1,mAP50,mAP50-95,inference_ms,FPS,method_label
0,day15_yolov8n_baseline-22,All validation,NaN,NaN,NaN,0.0,0.0,238.1767,4.1986,Day 15 - YOLOv8n baseline
1,day15_yolov8n_baseline-22,Low visibility,NaN,NaN,NaN,0.0,0.0,226.6165,4.4127,Day 15 - YOLOv8n baseline
2,day16_yolov8n_img832-28,All validation,NaN,NaN,NaN,0.0,0.0,226.7094,4.4109,Day 16 - YOLOv8n img832
3,day16_yolov8n_img832-28,Low visibility,NaN,NaN,NaN,0.0,0.0,236.7858,4.2232,Day 16 - YOLOv8n img832
4,day17_yolov8n_tuned_continuation,All validation,NaN,NaN,NaN,0.0,0.0,233.9658,4.2741,Day 17 - YOLOv8n tuned
5,day17_yolov8n_tuned_continuation,Low visibility,NaN,NaN,NaN,0.0,0.0,252.4192,3.9617,Day 17 - YOLOv8n tuned
6,day18_yolo11n_640-5,All validation,NaN,NaN,NaN,0.0,0.0,219.0262,4.5657,Day 18 - YOLO11n
7,day18_yolo11n_640-5,Low visibility,NaN,NaN,NaN,0.0,0.0,232.0514,4.3094,Day 18 - YOLO11n


In [17]:
# ============================================================
# 9. ROBUSTNESS EFFECT / CONDITION DROP
# ============================================================
# Quantify the change from all-validation performance to each condition.

base = OVERALL_RESULTS[OVERALL_RESULTS["condition"]=="All validation"].copy()

drop_rows=[]
for method in OVERALL_RESULTS["method_label"].unique():
    b = base[base["method_label"]==method]
    if b.empty: continue
    b = b.iloc[0]

    for condition in ["Low visibility","Small objects"]:
        c = OVERALL_RESULTS[
            (OVERALL_RESULTS["method_label"]==method) &
            (OVERALL_RESULTS["condition"]==condition)
        ]
        if c.empty: continue
        c=c.iloc[0]
        drop_rows.append({
            "method":method,
            "condition":condition,
            "mAP50_all":b["mAP50"],
            "mAP50_condition":c["mAP50"],
            "mAP50_change":c["mAP50"]-b["mAP50"],
            "mAP50-95_all":b["mAP50-95"],
            "mAP50-95_condition":c["mAP50-95"],
            "mAP50-95_change":c["mAP50-95"]-b["mAP50-95"],
            "precision_change":c["precision"]-b["precision"],
            "recall_change":c["recall"]-b["recall"],
            "F1_change":c["F1"]-b["F1"]
        })

ROBUSTNESS = pd.DataFrame(drop_rows)
ROBUSTNESS.to_csv(DAY20_ROOT/"day20_condition_effect.csv", index=False)

print("DAY 20 CONDITION EFFECT")
display(ROBUSTNESS.round(4))

DAY 20 CONDITION EFFECT


,method,condition,mAP50_all,mAP50_condition,mAP50_change,mAP50-95_all,mAP50-95_condition,mAP50-95_change,precision_change,recall_change,F1_change
0,Day 15 - YOLOv8n baseline,Low visibility,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,Day 16 - YOLOv8n img832,Low visibility,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2,Day 17 - YOLOv8n tuned,Low visibility,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
3,Day 18 - YOLO11n,Low visibility,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN


In [15]:
# ============================================================
# 10. CLASS-WISE DAY 20 TABLE
# ============================================================
class_table = CLASS_RESULTS[
    CLASS_RESULTS["condition"].isin(["All validation","Low visibility","Small objects"])
].copy()

class_table = class_table[
    ["method_label","condition","class","precision","recall","F1","mAP50-95","inference_ms","FPS"]
].sort_values(["method_label","condition","class"])

class_table.to_csv(DAY20_ROOT/"day20_classwise_table.csv", index=False)

display(class_table.round(4))

,method_label,condition,class,precision,recall,F1,mAP50-95,inference_ms,FPS
0,Day 15 - YOLOv8n baseline,All validation,trash_etc,NaN,NaN,NaN,0.0,238.1767,4.1986
1,Day 15 - YOLOv8n baseline,All validation,trash_fabric,NaN,NaN,NaN,0.0,238.1767,4.1986
2,Day 15 - YOLOv8n baseline,All validation,trash_fishing_gear,NaN,NaN,NaN,0.0,238.1767,4.1986
3,Day 15 - YOLOv8n baseline,All validation,trash_metal,NaN,NaN,NaN,0.0,238.1767,4.1986
4,Day 15 - YOLOv8n baseline,All validation,trash_paper,NaN,NaN,NaN,0.0,238.1767,4.1986
...,...,...,...,...,...,...,...,...,...
59,Day 18 - YOLO11n,Low visibility,trash_metal,NaN,NaN,NaN,0.0,232.0514,4.3094
60,Day 18 - YOLO11n,Low visibility,trash_paper,NaN,NaN,NaN,0.0,232.0514,4.3094
61,Day 18 - YOLO11n,Low visibility,trash_plastic,NaN,NaN,NaN,0.0,232.0514,4.3094
62,Day 18 - YOLO11n,Low visibility,trash_rubber,NaN,NaN,NaN,0.0,232.0514,4.3094


In [16]:
# ============================================================
# 11. SAVE DAY 20 RESEARCH INSIGHT
# ============================================================
insights = []

for method in OVERALL_RESULTS["method_label"].unique():
    m = OVERALL_RESULTS[OVERALL_RESULTS["method_label"]==method]

    def val(cond, col):
        x=m[m.condition==cond][col]
        return float(x.iloc[0]) if not x.empty else np.nan

    all_map=val("All validation","mAP50-95")
    low_map=val("Low visibility","mAP50-95")
    small_map=val("Small objects","mAP50-95")

    insights.append({
        "method":method,
        "all_val_mAP50_95":all_map,
        "low_visibility_mAP50_95":low_map,
        "small_object_mAP50_95":small_map,
        "low_visibility_change":low_map-all_map if np.isfinite(low_map) else np.nan,
        "small_object_change":small_map-all_map if np.isfinite(small_map) else np.nan
    })

INSIGHT_DF=pd.DataFrame(insights)
INSIGHT_DF.to_csv(DAY20_ROOT/"day20_research_insight.csv",index=False)

text = []
text.append("DAY 20 RESEARCH INSIGHT")
text.append("="*60)
text.append("The analysis uses the existing validation split only.")
text.append("The final test split was not used.")
text.append("")
text.append("Low visibility and small-object subsets are analysis proxies,")
text.append("not new ground-truth difficulty labels.")
text.append("")
for _,r in INSIGHT_DF.iterrows():
    text.append(f"Method: {r['method']}")
    text.append(f"  All validation mAP50-95: {r['all_val_mAP50_95']:.4f}")
    text.append(f"  Low visibility mAP50-95: {r['low_visibility_mAP50_95']:.4f}")
    text.append(f"  Small-object mAP50-95: {r['small_object_mAP50_95']:.4f}")
    text.append(f"  Low visibility change: {r['low_visibility_change']:.4f}")
    text.append(f"  Small-object change: {r['small_object_change']:.4f}")
    text.append("")

(DAY20_ROOT/"DAY20_RESEARCH_INSIGHT.txt").write_text("\n".join(text),encoding="utf-8")
print("\n".join(text))

DAY 20 RESEARCH INSIGHT
The analysis uses the existing validation split only.
The final test split was not used.

Low visibility and small-object subsets are analysis proxies,
not new ground-truth difficulty labels.

Method: Day 15 - YOLOv8n baseline
  All validation mAP50-95: 0.0000
  Low visibility mAP50-95: 0.0000
  Small-object mAP50-95: nan
  Low visibility change: 0.0000
  Small-object change: nan

Method: Day 16 - YOLOv8n img832
  All validation mAP50-95: 0.0000
  Low visibility mAP50-95: 0.0000
  Small-object mAP50-95: nan
  Low visibility change: 0.0000
  Small-object change: nan

Method: Day 17 - YOLOv8n tuned
  All validation mAP50-95: 0.0000
  Low visibility mAP50-95: 0.0000
  Small-object mAP50-95: nan
  Low visibility change: 0.0000
  Small-object change: nan

Method: Day 18 - YOLO11n
  All validation mAP50-95: 0.0000
  Low visibility mAP50-95: 0.0000
  Small-object mAP50-95: nan
  Low visibility change: 0.0000
  Small-object change: nan



## Day 20 output files

The notebook saves:

- `day20_overall_robustness.csv`
- `day20_classwise_robustness.csv`
- `day20_classwise_table.csv`
- `day20_condition_effect.csv`
- `day20_research_insight.csv`
- `DAY20_RESEARCH_INSIGHT.txt`
- `validation_condition_metadata.csv`

### Important interpretation
The low-visibility subset is based on brightness/contrast proxies, and the small-object subset is based on bounding-box area. These are **analysis conditions**, not manually assigned difficulty labels. The dataset observations in the project also identify underwater colour cast, turbidity/reduced visibility, clutter/overlap, and small objects as relevant challenges. fileciteturn47file0

No test images are used in this Day 20 notebook.